In [16]:
# osint_console_view.py — DDG + preferred sources + JSON persistence
import os, re, json, time, logging, warnings, argparse, textwrap, random, tempfile, shutil
from functools import lru_cache
from urllib.parse import urlparse, urlunparse, quote_plus
from typing import List, Tuple, Dict, Any, Optional
from collections import Counter

import requests
from bs4 import BeautifulSoup

# ===================== Settings =====================
ALLOW_INSECURE_SSL   = True
DEFAULT_TIMEOUT      = (6.0, 18.0)
MAX_RETRIES          = 3
SEARCH_LIMIT         = 15
PER_QUERY_PAUSE_S    = (0.6, 1.2)  # (low, high) jitter between searches

# Domains to avoid entirely (suffix match)
AVOID_DOMAINS = {"gogo.mn"}

# Consider intel portals "valid" even if short (toggle via --intel-valid)
INTEL_DOMAINS = (
    "otx.alienvault.com","abuseipdb.com","virustotal.com","urlscan.io",
    "censys.io","shodan.io","talosintelligence.com","threatfox","misp",
    "bazaar.abuse.ch","app.any.run","phishtank","greynoise.io","abuse.ch"
)

# Preferred intel sources for IPs
PREFERRED_SOURCES = {
    "abuseipdb":  lambda ip: f"https://www.abuseipdb.com/check/{ip}",
}

# quiet urllib3 warnings when using insecure SSL
if ALLOW_INSECURE_SSL:
    try:
        import urllib3
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
        warnings.filterwarnings('ignore', category=UserWarning, module='urllib3')
    except Exception:
        pass

# ===================== Logging ======================
log = logging.getLogger("osint_console_ddg")
if not log.handlers:
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    log.addHandler(h)
log.setLevel(logging.INFO)

# ===================== HTTP Session =================
UAS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_5) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.6 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
]

def build_session() -> requests.Session:
    s = requests.Session()
    s.trust_env = False  # ignore system proxy env unless you explicitly set proxies
    s.headers.update({
        'User-Agent': random.choice(UAS),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://duckduckgo.com/'
    })
    if ALLOW_INSECURE_SSL:
        s.verify = False
    return s

SESSION = build_session()

def rotate_ua():
    SESSION.headers["User-Agent"] = random.choice(UAS)

def backoff_sleep(attempt: int, base=0.8, cap=6.0):
    jitter = 0.5 + (os.urandom(1)[0] / 255.0)  # 0.5–1.5x
    time.sleep(min(cap, base * (2 ** attempt)) * jitter)

# ===================== CLI =========================
def get_args():
    parser = argparse.ArgumentParser(
        description="Console OSINT (DDG + AbuseIPDB/DNSlytics/VT). Prints cards and saves full data to JSON."
    )
    parser.add_argument("--term", action="append", help="Search term (repeatable).")
    parser.add_argument("--ip", help="IP address to query (shortcut for --term).")
    parser.add_argument("--limit", type=int, default=12, help="Max results to show per term after filtering.")
    parser.add_argument("--min-words", type=int, default=100, help="Minimum content words to count as valid.")
    parser.add_argument("--intel-valid", action="store_true", help="Count intel portals as valid even if short (requires IOCs).")
    parser.add_argument("--only-sources", action="store_true", help="Fetch only preferred intel sources (skip search).")
    parser.add_argument("--block", action="append", help="Domain suffix to block (e.g., example.com). Can repeat.")

    # Readability
    parser.add_argument("--width", type=int, default=100, help="Wrap width for text/excerpts (default 100).")
    parser.add_argument("--excerpt", type=int, default=400, help="Max excerpt characters shown per card (default 400).")
    parser.add_argument("--ioc-sample", type=int, default=3, help="IOCs listed per type before '+N more' (default 3).")
    parser.add_argument("--kw", type=int, default=8, help="How many keywords to show (default 8).")
    parser.add_argument("--no-color", action="store_true", help="Disable ANSI colors in output.")
    parser.add_argument("--tag", action="append", help="Add a tag string to all saved records (repeatable).")

    # Persistence
    parser.add_argument("--out", default=r"C:\Users\jaskew\Documents\project_repository\data\indicator library\indicator_library.json", help="JSON output file (default: indicator_library.json).")
    parser.add_argument("--reset-out", action="store_true", help="Truncate output file before writing.")
    parser.add_argument("--upsert", action="store_true", help="Upsert by URL (or source+URL for preferred sources) instead of blind append.")
    parser.add_argument("--no-content", action="store_true", help="Do not store full page content (store meta, IOCs, keywords only).")

    args, _ = parser.parse_known_args()
    return args

# ===================== URL utils ====================
def _host_suffix_match(host: str, banned: set) -> bool:
    host = (host or "").lower()
    return any(host == d or host.endswith("." + d) for d in banned)

def is_blocked_url(u: str) -> bool:
    try:
        h = urlparse(u).netloc
    except Exception:
        return False
    return _host_suffix_match(h, AVOID_DOMAINS)

def normalize_url(u: str) -> str:
    try:
        p = urlparse(u)
        scheme = p.scheme.lower()
        netloc = (p.hostname or "").lower()
        if p.port:
            default = (scheme == 'http' and p.port == 80) or (scheme == 'https' and p.port == 443)
            if not default:
                netloc = f"{netloc}:{p.port}"
        path = p.path or '/'
        if path != '/' and path.endswith('/'):
            path = path.rstrip('/')
        return urlunparse((scheme, netloc, path, '', p.query, ''))
    except Exception:
        return u

def is_ipv4(s: str) -> bool:
    if not re.match(r'^\d{1,3}(\.\d{1,3}){3}$', s or ""):
        return False
    try:
        return all(0 <= int(x) <= 255 for x in s.split('.'))
    except ValueError:
        return False


def _record_key(r: Dict[str, Any]) -> Tuple[str, str, str, str]:
    """(kind, term, url_normalized, source_or_blank) — stable unique key."""
    kind = r.get("kind") or ""
    term = r.get("term") or ""
    urln = r.get("url_normalized") or r.get("url") or ""
    src  = r.get("source") if kind == "preferred-source" else ""
    return (kind, term, urln, src or "")

def _upsert_in_list(data: List[Dict[str, Any]], rec: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Return a new list with `rec` upserted by canonical key."""
    key = _record_key(rec)
    out = []
    replaced = False
    for r in data:
        if _record_key(r) == key:
            out.append(rec)   # replace
            replaced = True
        else:
            out.append(r)
    if not replaced:
        out.append(rec)
    return out

def dedupe_indicator_file(path: str) -> None:
    """Load, collapse duplicates by canonical key (keep the latest by saved_at), write back."""
    data = _load_json_list(path)
    if not data:
        return
    best: Dict[Tuple[str,str,str,str], Dict[str,Any]] = {}
    for r in data:
        k = _record_key(r)
        # keep the newest by saved_at (fallback to append order)
        prev = best.get(k)
        if not prev:
            best[k] = r
        else:
            if (r.get("saved_at") or "") >= (prev.get("saved_at") or ""):
                best[k] = r
    _atomic_write_json(path, list(best.values()))

def _clean_comment(txt: str, max_len: int = 200) -> str:
    if not txt:
        return ""
    # strip boilerplate and emails/URLs
    txt = re.sub(r'\bshow\s+(more|less)\b', '', txt, flags=re.I)
    txt = re.sub(r'\S+@\S+', '', txt)                   # emails
    txt = re.sub(r'https?://\S+', '', txt)              # URLs
    txt = re.sub(r'\s+', ' ', txt).strip()              # collapse whitespace
    if len(txt) > max_len:
        txt = txt[:max_len-1] + "…"
    return txt

def _normalize_category_generic(cat: str) -> str:
    """No static list: trim, collapse spaces, simple Title-Case."""
    if not cat:
        return ""
    c = re.sub(r'\s+', ' ', cat).strip()
    # basic Title-Case; keeps existing uppercase chunks
    parts = []
    for token in c.split(' '):
        # if token is already mixed/upper (e.g., 'DDoS'), leave it
        if any(ch.isupper() for ch in token[1:]):
            parts.append(token)
        else:
            parts.append(token.capitalize())
    return ' '.join(parts)
# ===================== HTTP core (+ cache) =========
def http_get(url: str, retries=MAX_RETRIES, headers: Optional[dict]=None) -> Optional[requests.Response]:
    if is_blocked_url(url):
        log.info("Blocked by AVOID_DOMAINS: %s", url); return None
    last = None
    for i in range(retries):
        try:
            r = SESSION.get(url, timeout=DEFAULT_TIMEOUT, allow_redirects=True, headers=headers)
            if is_blocked_url(getattr(r, "url", url)): return None
            if r.status_code in (429, 503):
                last = r; log.warning("Throttled (%s). Backing off…", r.status_code); backoff_sleep(i); continue
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException as e:
            last = e; backoff_sleep(i)
    log.error("Failed to fetch %s: %s", url, last); return None

def ip_api_fallback(ip: str) -> Dict[str, Any]:
    """Very light ASN/rdns/country fallback via ip-api.com (no key, generous limits)."""
    try:
        r = SESSION.get(f"http://ip-api.com/json/{ip}?fields=status,country,as,org,reverse,query", timeout=DEFAULT_TIMEOUT)
        j = r.json() if r.status_code == 200 else {}
    except Exception:
        j = {}
    out = {}
    if (j or {}).get("status") == "success":
        # "as" comes like "ASxxxx FooBar", split number and name
        as_field = j.get("as") or ""
        m = re.match(r'AS(\d+)\s+(.*)', as_field)
        if m:
            out["asn"] = m.group(1)
            out["as_name"] = m.group(2)
        out["rdns"] = j.get("reverse") or None
        out["country"] = j.get("country") or None
        if j.get("org"): out["as_name"] = j["org"]
    return out


@lru_cache(maxsize=512)
def _cached_fetch(url: str) -> Optional[bytes]:
    r = http_get(url)
    return r.content if r else None

def fetch_html(url: str):
    raw = _cached_fetch(url)
    if raw is None:
        return None, None
    try:
        return raw.decode('utf-8', errors='replace'), 200
    except Exception:
        return None, 200

def soup_for(url: str) -> Tuple[Optional[BeautifulSoup], Optional[str]]:
    html, status = fetch_html(url)
    if html is None:
        return None, None
    return BeautifulSoup(html, 'html.parser'), html

# ===================== DuckDuckGo search ============
def ddg_query_variants(term: str) -> List[str]:
    qs = [f'"{term}"']
    if is_ipv4(term):
        qs += [
            f'"{term}" abuse', f'"{term}" report', f'"{term}" phishing"',
            f'"{term}" "command and control" OR c2',
            f'"{term}" blacklisted OR blocklist OR denylist'
        ]
    for d in AVOID_DOMAINS:
        qs = [f"{q} -site:{d}" for q in qs]
    return qs

def ddg_search(term: str, per_query_limit=SEARCH_LIMIT) -> List[Tuple[str,str]]:
    pairs: List[Tuple[str,str]] = []
    for q in ddg_query_variants(term):
        rotate_ua()
        url = f"https://html.duckduckgo.com/html/?q={quote_plus(q)}"
        soup, _ = soup_for(url)
        if not soup:
            continue
        out = []
        for a in soup.select("a.result__a, h2.result__title a, a.result__snippet"):
            href = a.get("href")
            title = a.get_text(strip=True)
            if not href or not title:
                continue
            if is_blocked_url(href):
                continue
            try:
                p = urlparse(href)
                if not p.scheme or not p.netloc:
                    continue
            except Exception:
                continue
            out.append((title, href))
            if len(out) >= per_query_limit:
                break
        pairs.extend(out)
        time.sleep(random.uniform(*PER_QUERY_PAUSE_S))

    # de-dup by normalized URL
    seen, uniq = set(), []
    for t, u in pairs:
        nu = normalize_url(u)
        if nu not in seen:
            seen.add(nu); uniq.append((t, u))
    return uniq

# ===================== Extraction ===================
def parse_jsonld_article(soup: BeautifulSoup):
    try:
        for tag in soup.find_all("script", type=lambda x: x and "ld+json" in x):
            txt = tag.string or tag.get_text()
            if not txt: continue
            data = json.loads(txt.strip())
            items = data if isinstance(data, list) else [data]
            for obj in items:
                if not isinstance(obj, dict): continue
                typ = obj.get("@type") or obj.get("@graph", [{}])[0].get("@type")
                if isinstance(typ, list):
                    is_article = any(isinstance(t, str) and t.lower().endswith("article") for t in typ)
                else:
                    is_article = isinstance(typ, str) and typ.lower().endswith("article")
                if is_article:
                    headline = obj.get("headline") or obj.get("name")
                    date_published = obj.get("datePublished") or obj.get("datepublished")
                    return headline, date_published
    except Exception:
        pass
    return None, None

def extract_metadata(url: str) -> Dict[str, Any]:
    raw = _cached_fetch(url)
    meta = {
        'status': 200 if raw is not None else None,
        'domain': None,
        'title_tag': None,
        'meta_description': None,
        'og_title': None,
        'og_description': None,
        'og_site_name': None,
        'article_published_time': None,
        'page_text_excerpt': None
    }
    try:
        meta['domain'] = urlparse(url).netloc
    except Exception:
        meta['domain'] = None
    if raw is None:
        return meta
    try:
        soup = BeautifulSoup(raw, 'html.parser')
        tt = soup.find('title')
        if tt: meta['title_tag'] = tt.get_text(strip=True)
        md = soup.find('meta', attrs={'name': 'description'})
        if md and md.get('content'): meta['meta_description'] = md['content'].strip()
        ogt = soup.find('meta', property='og:title')
        if ogt and ogt.get('content'): meta['og_title'] = ogt.get('content').strip()
        ogd = soup.find('meta', property='og:description')
        if ogd and ogd.get('content'): meta['og_description'] = ogd.get('content').strip()
        ogs = soup.find('meta', property='og:site_name')
        if ogs and ogs.get('content'): meta['og_site_name'] = ogs.get('content').strip()
        ap = soup.find('meta', property='article:published_time')
        if ap and ap.get('content'): meta['article_published_time'] = ap.get('content').strip()

        _, jsonld_date = parse_jsonld_article(soup)
        if jsonld_date and not meta['article_published_time']:
            meta['article_published_time'] = jsonld_date

        body_text = soup.get_text(separator=' ', strip=True)
        if body_text:
            meta['page_text_excerpt'] = ' '.join(body_text.split()[:200])
    except Exception:
        pass
    return meta

def extract_article_text(url: str) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    raw = _cached_fetch(url)
    if not raw: return None, None, None
    soup = BeautifulSoup(raw, 'html.parser')
    article_div = soup.find('article') or soup.find('div', class_='article-body') or soup.find('div', class_='content')
    text = article_div.get_text(separator=' ', strip=True) if article_div else soup.get_text(separator=' ', strip=True)
    text = re.sub(r'\s+', ' ', (text or '')).strip()
    _, jsonld_date = parse_jsonld_article(soup)
    ap = soup.find('meta', property='article:published_time')
    publish_date = (ap.get('content').strip() if ap and ap.get('content') else jsonld_date) or "Unknown"
    summary = ' '.join(text.split()[:50]) if text else None
    return text, summary, publish_date

def extract_iocs(text: str) -> Dict[str,Any]:
    if not text:
        return {}
    ipv4   = re.findall(r'\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b', text)
    domain = re.findall(r'\b(?!(?:\d+\.){3}\d+)(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}\b', text)
    url    = re.findall(r'\bhttps?://[^\s"\)\]\}<>]+', text)
    email  = re.findall(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', text)
    md5    = re.findall(r'\b[a-fA-F0-9]{32}\b', text)
    sha1   = re.findall(r'\b[a-fA-F0-9]{40}\b', text)
    sha256 = re.findall(r'\b[a-fA-F0-9]{64}\b', text)

    def uniq(seq):
        seen, out = set(), []
        for x in seq:
            if x not in seen: seen.add(x); out.append(x)
        return out

    return {
        'ips': uniq(ipv4),
        'domains': uniq(domain),
        'urls': uniq(url),
        'emails': uniq(email),
        'hashes': {'md5': uniq(md5), 'sha1': uniq(sha1), 'sha256': uniq(sha256)}
    }

def extract_keywords(text: str, top_n=10) -> List[str]:
    if not text:
        return []
    tokens = re.findall(r"[A-Za-z]{4,}", text.lower())
    freq: Dict[str,int] = {}
    for t in tokens:
        freq[t] = freq.get(t, 0) + 1
    return [w for w,_ in sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:top_n]]

# ===================== Preferred sources ============
def build_source_urls_for_ip(ip: str):
    return [(name, fn(ip)) for name, fn in PREFERRED_SOURCES.items()]

def parse_abuseipdb(html: str) -> Dict[str, Any]:
    soup = BeautifulSoup(html, 'html.parser')
    out: Dict[str, Any] = {}

    # quick stats (unchanged)
    text = soup.get_text(" ", strip=True)
    m = re.search(r'(?i)Abuse\s+Confidence\s+Score[^0-9]*?(\d+)\s*%', text)
    if m: out['abuse_confidence_score'] = int(m.group(1))
    m = re.search(r'(?i)Total\s+Reports:\s*([\d,]+)', text)
    if m: out['total_reports'] = int(m.group(1).replace(',', ''))

    # --- pull table, produce a local list only (no storing) ---
    reports_min_local = []
    target_table = None
    for tbl in soup.select('table'):
        headers = [th.get_text(strip=True).lower() for th in tbl.select('thead th')]
        if 'comment' in headers and 'categories' in headers:
            target_table = tbl
            break

    if target_table:
        headers = [th.get_text(strip=True).lower() for th in target_table.select('thead th')]
        idx_comment = headers.index('comment')
        idx_cats    = headers.index('categories')
        for tr in target_table.select('tbody tr'):
            tds = tr.find_all('td')
            if len(tds) <= max(idx_comment, idx_cats): 
                continue
            comment = tds[idx_comment].get_text(" ", strip=True)
            cats_raw = [el.get_text(strip=True) for el in tds[idx_cats].select('span, a, div')]
            cats = [c for c in cats_raw if c]
            if comment or cats:
                reports_min_local.append({'comment': comment, 'categories': cats})

    # --- your existing cleaners (generic) ---
    seen = set()
    flat = []
    from collections import Counter
    cat_counter = Counter()

    def _clean_comment(txt: str, max_len: int = 200) -> str:
        if not txt: return ""
        txt = re.sub(r'\bshow\s+(more|less)\b', '', txt, flags=re.I)
        txt = re.sub(r'\S+@\S+', '', txt)
        txt = re.sub(r'https?://\S+', '', txt)
        txt = re.sub(r'\s+', ' ', txt).strip()
        return (txt[:max_len-1] + "…") if len(txt) > max_len else txt

    def _normalize_category_generic(cat: str) -> str:
        if not cat: return ""
        c = re.sub(r'\s+', ' ', cat).strip()
        parts = []
        for token in c.split(' '):
            if any(ch.isupper() for ch in token[1:]):
                parts.append(token)
            else:
                parts.append(token.capitalize())
        return ' '.join(parts)

    for r in reports_min_local:
        cats_norm = []
        for c in r.get('categories', []):
            cn = _normalize_category_generic(c)
            if cn:
                cats_norm.append(cn); cat_counter[cn] += 1
        cats_norm = list(dict.fromkeys(cats_norm))
        cmnt = _clean_comment(r.get('comment', ''))
        line = f"[{', '.join(cats_norm)}] {cmnt}" if cats_norm else (cmnt or "")
        key = (tuple(cats_norm), cmnt)
        if line and key not in seen:
            seen.add(key); flat.append(line)

    out['reports_flat'] = flat
    if cat_counter:
        out['category_counts'] = dict(cat_counter.most_common())

    # explicitly ensure we DO NOT include reports_min
    # (defensive if you later refactor)
    out.pop('reports_min', None)
    return out


def fetch_preferred_sources(ip: str, limit: int = 3) -> List[Tuple[str, str, Dict[str, Any], Dict[str, Any]]]:
    results = []
    for name, url in build_source_urls_for_ip(ip)[:limit]:
        raw = http_get(url)
        if not raw:
            continue
        meta = {
            'status': raw.status_code,
            'domain': urlparse(url).netloc,
            'title_tag': None,
            'article_published_time': None,
            'page_text_excerpt': None
        }
        soup = BeautifulSoup(raw.content, 'html.parser')
        tt = soup.find('title')
        if tt:
            meta['title_tag'] = tt.get_text(strip=True)
        body_text = soup.get_text(separator=' ', strip=True)
        if body_text:
            meta['page_text_excerpt'] = ' '.join(body_text.split()[:200])

        html = raw.text
        if name == "abuseipdb":
            parsed = parse_abuseipdb(html)
        else:
            parsed = {}
        results.append((name, url, meta, parsed))
    return results

# ================ Pretty print helpers ==============
try:
    import colorama
    colorama.just_fix_windows_console()
    _HAS_COLORAMA = True
except Exception:
    _HAS_COLORAMA = False

class Palette:
    def __init__(self, enabled=True):
        self.enabled = enabled
        self.dim   = "\x1b[2m" if enabled else ""
        self.bold  = "\x1b[1m" if enabled else ""
        self.red   = "\x1b[31m" if enabled else ""
        self.green = "\x1b[32m" if enabled else ""
        self.yellow= "\x1b[33m" if enabled else ""
        self.blue  = "\x1b[34m" if enabled else ""
        self.mag   = "\x1b[35m" if enabled else ""
        self.cyan  = "\x1b[36m" if enabled else ""
        self.reset = "\x1b[0m"  if enabled else ""

def _wrap(text, width, indent=4):
    tw = textwrap.TextWrapper(width=width, subsequent_indent=" " * indent, break_long_words=False, break_on_hyphens=False)
    for line in tw.wrap(text):
        yield line

def _truncate(s, n):
    if not s: return ""
    return (s[:n-1] + "…") if len(s) > n else s

def _fmt_kv_row(items, sep="  |  "):
    return sep.join(f"{k}: {v}" for k, v in items if v not in (None, "", []))

def _sample_list(lst, k):
    if not lst: return ""
    lst = list(dict.fromkeys(lst))  # dedupe keep order
    if len(lst) <= k:
        return ", ".join(lst)
    return ", ".join(lst[:k]) + f"  (+{len(lst)-k} more)"

# ===================== Console UI ==================
def print_source_card(idx: int, name: str, url: str, meta: Dict[str,Any], parsed: Dict[str,Any], *, width=100, pal=None, excerpt_chars=400):
    pal = pal or Palette(False)
    border = "=" * width
    print(border)
    head = f"[{idx}] [{name}] {url}"
    print(f"{pal.bold}{head}{pal.reset}")

    facts = [
        ("Domain", meta.get('domain')),
        ("HTTP", meta.get('status')),
        ("Title", _truncate(meta.get('title_tag') or "", max(30, width//3))),
    ]
    print("    " + _fmt_kv_row(facts))

    summary = []
    if "abuse_confidence_score" in parsed:
        score = parsed["abuse_confidence_score"]
        color = pal.red if isinstance(score, int) and score >= 50 else pal.yellow if isinstance(score, int) and score >= 10 else pal.green
        summary.append(f"ACS: {color}{score}%{pal.reset}")
    if "total_reports" in parsed:
        summary.append(f"Reports: {parsed['total_reports']}")
    for k in ("asn","as_name","country","rdns","ip_range"):
        if k in parsed:
            label = k.replace("_"," ").upper()
            summary.append(f"{label}: {_truncate(str(parsed[k]), 40)}")
    if "detected_by_engines" in parsed:
        summary.append(f"VT Eng: {parsed['detected_by_engines']}")
    if summary:
        print("    " + "  |  ".join(summary))

    if parsed.get("tags"):
        print("    Tags: " + _sample_list(parsed["tags"], 10))
    if parsed.get("hostnames"):
        print("    Hostnames: " + _sample_list(parsed["hostnames"], 8))

    rows = (parsed or {}).get('reports_flat') or []
    if rows:
        print("    Recent reports:")
        for line in rows[:5]:  # or your --report-sample
            print(f"      - {line}")

def print_card(idx: int, title: str, url: str, meta: Dict[str,Any], text: Optional[str],
               publish_date: Optional[str], keywords: List[str], iocs: Dict[str,Any],
               *, width=100, pal=None, excerpt_chars=400, kw_n=8, ioc_k=3):
    pal = pal or Palette(False)
    border = "-" * width
    excerpt = (text or meta.get('page_text_excerpt') or "")
    excerpt = _truncate(excerpt, excerpt_chars)
    dom  = meta.get('domain')
    stat = meta.get('status')
    words = (len(text.split()) if text else 0)

    print(border)
    print(f"{pal.bold}[{idx}] {title}{pal.reset}")
    print(f"    URL: {url}")

    facts = [
        ("Domain", dom),
        ("HTTP", stat),
        ("Words", words),
        ("Published", publish_date or meta.get('article_published_time') or "Unknown"),
    ]
    print("    " + _fmt_kv_row(facts))
    if keywords:
        print("    Keywords: " + ", ".join(keywords[:kw_n]))

    def _fmt_ioc(name, seq):
        if not seq: return None
        return f"{name} [{len(seq)}]: " + _sample_list(seq, ioc_k)

    ioc_lines = [
        _fmt_ioc("IPs", iocs.get("ips")),
        _fmt_ioc("Domains", iocs.get("domains")),
        _fmt_ioc("URLs", iocs.get("urls")),
        _fmt_ioc("Emails", iocs.get("emails")),
    ]
    hashes = iocs.get("hashes") or {}
    for hname in ("md5","sha1","sha256"):
        line = _fmt_ioc(hname.upper(), hashes.get(hname))
        if line: ioc_lines.append(line)
    ioc_lines = [x for x in ioc_lines if x]
    if ioc_lines:
        for line in ioc_lines:
            for pl in _wrap(line, width=width, indent=10):
                print("    " + pl)
    if excerpt:
        for line in _wrap(excerpt, width=width, indent=4):
            print("    " + line)

def is_intel_like(domain: Optional[str]) -> bool:
    if not domain: return False
    d = domain.lower()
    return any(x in d for x in INTEL_DOMAINS)

# ===================== JSON persistence =============
def _load_json_list(path: str) -> List[Dict[str,Any]]:
    if not os.path.exists(path):
        return []
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            return data if isinstance(data, list) else []
    except Exception:
        return []


def _atomic_write_json(path: str, data: List[Dict[str, Any]]) -> None:
    """Atomically write a JSON list to disk (create parent dir if needed)."""
    d = os.path.dirname(os.path.abspath(path)) or "."
    os.makedirs(d, exist_ok=True)

    fd, tmp = tempfile.mkstemp(prefix="indlib_", suffix=".json", dir=d)
    os.close(fd)
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        # os.replace handles overwrite atomically on Windows & POSIX
        os.replace(tmp, path)
    finally:
        # best-effort cleanup if something went wrong
        try:
            if os.path.exists(tmp):
                os.remove(tmp)
        except Exception:
            pass


def indicator_exists(
    data: List[Dict[str, Any]],
    *,
    kind: str,
    url_normalized: str,
    term: str,
    source: Optional[str] = None
) -> bool:
    """Check if a record already exists. For preferred-source, also match `source`."""
    for r in data:
        if r.get("kind") != kind:
            continue
        if r.get("url_normalized") != url_normalized or r.get("term") != term:
            continue
        if kind == "preferred-source":
            return r.get("source") == source
        return True
    return False

def _now_iso() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

def _base_record(term: str, kind: str, url: str, tags: Optional[List[str]]) -> Dict[str,Any]:
    return {
        "term": term,
        "kind": kind,  # "preferred-source" or "search-result"
        "saved_at": _now_iso(),
        "url": url,
        "url_normalized": normalize_url(url),
        "tags": tags or []
    }

def save_preferred_record(out_path: str, _upsert_flag: bool, term: str, name: str, url: str,
                          meta: Dict[str,Any], parsed: Dict[str,Any], tags: Optional[List[str]]):
    rec = _base_record(term, "preferred-source", url, tags)
    rec.update({"source": name, "meta": meta, "parsed": parsed})
    data = _load_json_list(out_path)
    data = _upsert_in_list(data, rec)
    _atomic_write_json(out_path, data)

def save_search_record(out_path: str, _upsert_flag: bool, term: str, title: str, url: str,
                       meta: Dict[str,Any], text: Optional[str], publish_date: Optional[str],
                       keywords: List[str], iocs: Dict[str,Any], tags: Optional[List[str]], no_content: bool):
    rec = _base_record(term, "search-result", url, tags)
    rec.update({
        "title": title,
        "meta": meta,
        "publish_date": publish_date or meta.get("article_published_time"),
        "keywords": keywords,
        "iocs": iocs,
    })
    if not no_content:
        rec["content"] = text
        rec["excerpt"] = (text or meta.get('page_text_excerpt') or "")[:2000]
    data = _load_json_list(out_path)
    data = _upsert_in_list(data, rec)
    _atomic_write_json(out_path, data)


def save_search_record(out_path: str, _upsert_flag: bool, term: str, title: str, url: str,
                       meta: Dict[str,Any], text: Optional[str], publish_date: Optional[str],
                       keywords: List[str], iocs: Dict[str,Any], tags: Optional[List[str]], no_content: bool):
    rec = _base_record(term, "search-result", url, tags)
    rec.update({
        "title": title,
        "meta": meta,
        "publish_date": publish_date or meta.get("article_published_time"),
        "keywords": keywords,
        "iocs": iocs,
    })
    if not no_content:
        rec["content"] = text
        rec["excerpt"] = (text or meta.get('page_text_excerpt') or "")[:2000]
    data = _load_json_list(out_path)
    data = _upsert_in_list(data, rec)
    _atomic_write_json(out_path, data)

# ===================== Main ========================
def main(args, pal):
    # blocklist extend
    if args.block:
        for d in args.block:
            if d: AVOID_DOMAINS.add(d.lower())

    # handle reset-out
    if args.reset_out and os.path.exists(args.out):
        _atomic_write_json(args.out, [])

    terms = []
    if args.ip:
        terms.append(args.ip)
    if args.term:
        terms.extend(args.term)
    if not terms:
        # Load indicators from CSV and use as terms
        import csv
        indicators_csv = r"C:\Users\jaskew\Documents\project_repository\data\indicator library\indicators.csv"
        terms = []
        with open(indicators_csv, newline='', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile)
            for row in reader:
                if row and row[0].strip():
                    terms.append(row[0].strip())
        if not terms:
            print(f"No indicators found in {indicators_csv}")
            return

    # Load existing indicators from JSON
    indicator_file = args.out
    existing_data = _load_json_list(indicator_file)
    existing_terms = set()
    for rec in existing_data:
        t = rec.get("term")
        if t:
            existing_terms.add(t)

    # Filter out terms that already exist
    original_count = len(terms)
    terms = [t for t in terms if t not in existing_terms]
    if not terms:
        print("All indicators already exist in the indicator library. Nothing to search.")
        return
    elif len(terms) < original_count:
        print(f"Skipped {original_count - len(terms)} indicators already present in the indicator library.")

    log.info("trust_env=%s, session.verify=%s, blocked=%s, out=%s",
             SESSION.trust_env, getattr(SESSION, 'verify', None), ",".join(sorted(AVOID_DOMAINS)), args.out)

    for term in terms:
        print(f"\n>>> Searching for: {term}")

        shown = 0
        idx = 1

        # 1) Preferred sources for IPs
        if is_ipv4(term) and args.only_sources:
            src_results = fetch_preferred_sources(term)
            if src_results:
                for name, url, meta, parsed in src_results:
                    print_source_card(idx, name, url, meta, parsed,
                                      width=args.width, pal=pal, excerpt_chars=args.excerpt)
                    # persist
                    save_preferred_record(args.out, args.upsert, term, name, url, meta, parsed, args.tag)
                    print()
                    idx += 1; shown += 1
            if shown == 0:
                print("No data returned from preferred sources.")
            continue

        if is_ipv4(term):
            for name, url, meta, parsed in fetch_preferred_sources(term):
                print_source_card(idx, name, url, meta, parsed,
                                  width=args.width, pal=pal, excerpt_chars=args.excerpt)
                save_preferred_record(args.out, args.upsert, term, name, url, meta, parsed, args.tag)
                print()
                idx += 1; shown += 1

        # 2) DDG search
        pairs = ddg_search(term, per_query_limit=SEARCH_LIMIT)
        print(f"Collected {len(pairs)} unique links via DDG")

        for title, link in pairs:
            if is_blocked_url(link):
                continue
            meta = extract_metadata(link)
            text, summary, publish_date = extract_article_text(link)
            iocs = extract_iocs(text or meta.get('page_text_excerpt') or "")
            keywords = extract_keywords((text or summary or meta.get('page_text_excerpt') or "")[:5000])

            valid = (len(text.split()) if text else 0) >= args.min_words
            if not valid and args.intel_valid:
                if is_intel_like(meta.get('domain')) and iocs and any(iocs.values()):
                    valid = True
            if not valid:
                # still store? You asked for "full information"—we’ll store everything even if not shown.
                save_search_record(args.out, args.upsert, term, title, link, meta, text, publish_date, keywords, iocs, args.tag, args.no_content)
                continue

            print_card(idx, title, link, meta, text, publish_date, keywords, iocs,
                       width=args.width, pal=pal, excerpt_chars=args.excerpt,
                       kw_n=args.kw, ioc_k=args.ioc_sample)
            # persist
            save_search_record(args.out, args.upsert, term, title, link, meta, text, publish_date, keywords, iocs, args.tag, args.no_content)
            print()
            idx += 1; shown += 1
            if shown >= args.limit:
                break

        if shown == 0:
            msg = "No valid items met the criteria."
            if is_ipv4(term):
                msg += " Try --only-sources or lower --min-words."
            print(msg)

if __name__ == "__main__":
    args = get_args()
    pal = Palette(enabled=(not args.no_color))
    main(args, pal)


2025-09-25 14:37:10,958 | INFO | trust_env=False, session.verify=False, blocked=gogo.mn, out=C:\Users\jaskew\Documents\project_repository\data\indicator library\indicator_library.json


Skipped 141 indicators already present in the indicator library.

>>> Searching for: indicator


2025-09-25 14:37:34,350 | ERROR | Failed to fetch https://html.duckduckgo.com/html/?q=%22indicator%22+-site%3Agogo.mn: HTTPSConnectionPool(host='html.duckduckgo.com', port=443): Max retries exceeded with url: /html/?q=%22indicator%22+-site%3Agogo.mn (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000000000CF7DA90>, 'Connection to html.duckduckgo.com timed out. (connect timeout=6.0)'))


Collected 0 unique links via DDG
No valid items met the criteria.

>>> Searching for: 104.28.155.218
[1] [abuseipdb] https://www.abuseipdb.com/check/104.28.155.218
    Domain: www.abuseipdb.com  |  HTTP: 200  |  Title: 104.28.155.218 | Cloudflare, Inc…
    Recent reports:
      - [Port Scan] Ports: 24x 27329
      - [DDoS Attack, Web Spam] Incoming Layer 7 Flood Detected
      - [Web App Attack] Wordpress Vunerability attack
      - [Brute-Force, SSH] Automated DDoS behavior detected targeting production services. Multiple anomalous connections and p ... Automated DDoS behavior detected targeting production services. Multiple anomalous connections…
      - [Brute-Force] list.rtbh.com.tr report: tcp/0



2025-09-25 14:37:58,880 | ERROR | Failed to fetch https://html.duckduckgo.com/html/?q=%22104.28.155.218%22+-site%3Agogo.mn: HTTPSConnectionPool(host='html.duckduckgo.com', port=443): Max retries exceeded with url: /html/?q=%22104.28.155.218%22+-site%3Agogo.mn (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000000000CF7E0D0>, 'Connection to html.duckduckgo.com timed out. (connect timeout=6.0)'))
2025-09-25 14:38:23,561 | ERROR | Failed to fetch https://html.duckduckgo.com/html/?q=%22104.28.155.218%22+abuse+-site%3Agogo.mn: HTTPSConnectionPool(host='html.duckduckgo.com', port=443): Max retries exceeded with url: /html/?q=%22104.28.155.218%22+abuse+-site%3Agogo.mn (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000000000A632E90>, 'Connection to html.duckduckgo.com timed out. (connect timeout=6.0)'))
2025-09-25 14:38:47,724 | ERROR | Failed to fetch https://html.duckduckgo.com/html/?q=%22104.28.155.218%22+report+-site%3Agogo.mn:

...

2025-09-25 14:58:30,238 | WARNING | Throttled (429). Backing off…


Collected 0 unique links via DDG

>>> Searching for: 122.54.133.163
[1] [abuseipdb] https://www.abuseipdb.com/check/122.54.133.163
    Domain: www.abuseipdb.com  |  HTTP: 200  |  Title: 122.54.133.163 | IPG | AbuseIPDB
    Recent reports:
      - [DDoS Attack] Malicious activity detected from 9299 IPG-AS-AP Philippine Long Distance Telephone Company towards h ... Malicious activity detected from 9299 IPG-AS-AP Philippine Long Distance Telephone Company tow…
      - [DDoS Attack, Web Spam] Incoming Layer 7 Flood Detected
      - [DDoS Attack, Hacking, IoT Targeted] TCP Watch Auto Report: Detected a ddos attack and suspicious activity from this IP, indicating a pot ... TCP Watch Auto Report: Detected a ddos attack and suspicious activity from this IP, indicating…
      - [DDoS Attack] Detected L7 DDoS via Cloudflare

Collected 0 unique links via DDG

>>> Searching for: 123.150.138.196
[1] [abuseipdb] https://www.abuseipdb.com/check/123.150.138.196
    Domain: www.abuseipdb.com  |  HTTP: 

2025-09-25 14:59:19,511 | ERROR | Failed to fetch https://html.duckduckgo.com/html/?q=%22123.58.209.112%22+abuse+-site%3Agogo.mn: 403 Client Error: Forbidden for url: https://html.duckduckgo.com/html/?q=%22123.58.209.112%22+abuse+-site%3Agogo.mn
2025-09-25 14:59:27,184 | ERROR | Failed to fetch https://html.duckduckgo.com/html/?q=%22123.58.209.112%22+blacklisted+OR+blocklist+OR+denylist+-site%3Agogo.mn: 403 Client Error: Forbidden for url: https://html.duckduckgo.com/html/?q=%22123.58.209.112%22+blacklisted+OR+blocklist+OR+denylist+-site%3Agogo.mn


Collected 0 unique links via DDG

>>> Searching for: 124.156.231.182
[1] [abuseipdb] https://www.abuseipdb.com/check/124.156.231.182
    Domain: www.abuseipdb.com  |  HTTP: 200  |  Title: 124.156.231.182 | Asia Pacific N…
    Recent reports:
      - [Brute-Force, Web App Attack] 124.156.231.182 - - [25/Sep/2025:13:26:06 +0200] "GET /containers/json HTTP/1.1" 404 162 "-" "Mozill ... 124.156.231.182 - - [25/Sep/2025:13:26:06 +0200] "GET /containers/json HTTP/1.1" 404 162 "-" "…
      - [DDoS Attack, Port Scan, Hacking, Brute-Force, Web App Attack, SSH] Automatic report from iptables firewall - detected malicious activity
      - [Port Scan, Web App Attack] 2025-09-24 21:53:36 UTC Unauthorized activity to TCP port 8080. Web App
      - [Web App Attack] Fail2ban picked up 124.156.231.182 attacking nginx
      - [Hacking, Bad Web Bot, Web App Attack] $f2bV_matches



KeyboardInterrupt: 